In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
from model.expression_embedding import expression_GVAE
from common.data_processing import data_preprocessing_ST1K

from Hist2ST.HIST2ST_train import main

In [ ]:
# input_path='./data/HEST/SKIN-LP-NL/'
# adata_list, HE_image_paths,nuclei_mask_paths=data_preprocessing_ST1K(input_path)
# names = [d[:-5] for d in os.listdir(input_path) if d.endswith('.h5ad')]
input_path='./data/HEST/COLON-CANCER_Xenium/'
names = [d[:-5] for d in os.listdir(input_path) if d.endswith('.h5ad')]
adata_list, HE_image_paths, nuclei_mask_paths=data_preprocessing_ST1K(input_path)
print(names)
print(len(names))
print(len(adata_list))
print(len(HE_image_paths))

In [ ]:

for i in range(len(adata_list)):
    sc.pp.filter_genes(adata_list[i], min_cells=1)
    sc.pp.filter_cells(adata_list[i], min_genes=1)
# 计算所有anndata共享的var_names
shared_var_names = set(adata_list[0].var_names)
for adata in adata_list[1:]:
    shared_var_names.intersection_update(adata.var_names)
print(len(shared_var_names))

for i in range(len(adata_list)):
    adata_list[i] = adata_list[i][:, list(shared_var_names)]
    
print(adata_list[0])

In [ ]:
Input_Method='hvg'
new_adata_list=expression_GVAE(adatas=adata_list,method='GATE',feature_method=Input_Method,n_top_genes=100)
print(new_adata_list[0])

In [ ]:
out_embedding=new_adata_list[0].X.shape[1]
for i in range(len(new_adata_list)):
    if 'block_r' in adata_list[i].uns and adata_list[i].uns['block_r'] > 32:
        new_adata_list[i].uns['block_r']=adata_list[i].uns['block_r']
        print(new_adata_list[i].uns['block_r'])
    else:
        new_adata_list[i].uns['block_r']=112
print(out_embedding)

In [ ]:
train_index=[i for i in range(len(new_adata_list))]
for i in train_index: 
    test_index=[i]
    new_train_index = [item for item in train_index if item not in test_index]
    savepath=os.path.join(input_path,names[test_index[0]],'results/')
    if not os.path.exists(savepath):
        os.makedirs(savepath)
    pred, gt = main(new_adata_list,HE_image_paths,names,new_train_index,test_index,out_embedding)
    pred.write_h5ad(os.path.join(savepath,'Hist2ST_pre.h5ad'))
    gt.write_h5ad(os.path.join(savepath,'Hist2ST_gt.h5ad'))